In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, HashingTF, IDF


In [3]:
D1 = "medical image classification"
D2 = "medical image analysis"
D3 = "natural language processing"
Q = "medical image classification"

In [4]:
# Khởi tạo Spark Session
spark = SparkSession.builder.appName("IDFCalculation").getOrCreate()

# Chuẩn bị dữ liệu từ các biến đã có
data = [(0, D1), (1, D2), (2, D3)]
df = spark.createDataFrame(data, ["id", "sentence"])

# Tokenize văn bản
tokenizer = Tokenizer(inputCol="sentence", outputCol="words")
wordsData = tokenizer.transform(df)

# Tính raw count vector
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=7)
featurizedData = hashingTF.transform(wordsData)

# Tính IDF
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurizedData)
rescaledData = idfModel.transform(featurizedData)

In [5]:
# Hiển thị kết quả IDF cho các từ
print("IDF Vector cho mỗi tài liệu:")
rescaledData.select("id", "features").show(truncate=False)

# Hiển thị giá trị IDF của mô hình
print("Giá trị IDF của các thuộc tính (terms):")
print(idfModel.idf)

IDF Vector cho mỗi tài liệu:
+---+-------------------------------------------------------------------------+
|id |features                                                                 |
+---+-------------------------------------------------------------------------+
|0  |(7,[0,1,6],[0.28768207245178085,0.28768207245178085,0.6931471805599453]) |
|1  |(7,[0,1,2],[0.28768207245178085,0.28768207245178085,0.28768207245178085])|
|2  |(7,[2,3],[0.28768207245178085,1.3862943611198906])                       |
+---+-------------------------------------------------------------------------+

Giá trị IDF của các thuộc tính (terms):
[0.28768207245178085,0.28768207245178085,0.28768207245178085,0.6931471805599453,1.3862943611198906,1.3862943611198906,0.6931471805599453]
